# TASK
Implement the core training loop of word2vec in pure NumPy (no PyTorch / TensorFlow or other ML frameworks). The applicant is free to choose any suitable text dataset. The task is to implement the optimization procedure (forward pass, loss, gradients, and parameter updates) for a standard word2vec variant (e.g. skip-gram with negative sampling or CBOW).
The submitted solution should be fully understood by the applicant: during follow-up we will ask questions about the code, gradient derivation, and possible alternative implementations or optimizations.
Preferably, solutions should be provided as a link to a public GitHub repository.

## Chosen approach

I went with **skip-gram + negative sampling (SGNS)**.
Instead of computing a full softmax over the entire vocabulary (expensive), we only contrast the true context word against a small handful of randomly sampled 'noise' words. This makes each training step cheap and the model scales to large corpora easily.

Choosen datest is **text8**:  the first 10^9 bytes of the English Wikipedia (03.03.2006). It's a standard benchmark corpus for word embeddings and small enough to train on a CPU in a reasonable time. The file is downloaded once and cached locally. Skip-gram tends to produce better embeddings for rare words compared to CBOW, which matters here since text8 has a long tail.

## Imports

In [1]:
import os, re, time, zipfile, urllib.request, warnings
from collections import Counter
from dataclasses import dataclass
from typing import List, Tuple, Optional

import numpy as np
from tqdm.auto import tqdm

## Config

The defaults follow Mikolov et al. (2013).

In [2]:
@dataclass
class Config:
    # Corpus
    vocab_size:   int   = 30_000
    min_count:    int   = 5
    subsample_t:  float = 1e-5
    # Model
    embed_dim:    int   = 100
    window:       int   = 5
    k_negatives:  int   = 5
    neg_power:    float = 0.75
    # Training
    epochs:       int   = 5
    batch_size:   int   = 512
    # Adam
    lr:           float = 0.001
    beta1:        float = 0.9
    beta2:        float = 0.999
    adam_eps:     float = 1e-8
    # Misc
    seed:         int   = 42

## Data - text8

In [3]:
def fetch_text8(dest="text8.zip") -> List[str]:
    url = "http://mattmahoney.net/dc/text8.zip"
    if not os.path.exists(dest):
        print("Downloading text8 (~30 MB) …")
        urllib.request.urlretrieve(url, dest)
    with zipfile.ZipFile(dest) as zf:
        return zf.read("text8").decode().split()


## Vocabulary

This class:

1. **Truncate the vocabulary** — keep only the top-N most frequent words and drop anything that appears fewer than `min_count` times.
2. **Subsample frequent words** — words like *the*, "a" and *of* appear so often that they'd dominate training without adding much information. We drop each token with probability `1 − √(t / f(w))`, which roughly equalises the effective frequency across the vocabulary (Mikolov 2013 §2.3).
3. **Build the negative-sampling table** — a large integer array where word `i` occupies a fraction proportional to `freq(i)^0.75`. Sampling a negative is then just a single random index lookup — O(1) and cache-friendly.

In [4]:
class Vocabulary:
    """
    Builds word-index mappings, computes the negative-sampling table,
    and applies frequency-based subsampling to the corpus.
    """

    NEG_TABLE_SIZE = 1_000_000   # pre-built lookup table for O(1) sampling

    def __init__(self, cfg: Config):
        self.cfg = cfg

    def build(self, tokens: List[str]) -> np.ndarray:
        cfg = self.cfg
        freq = Counter(tokens)
        kept = [(w, c) for w, c in freq.most_common(cfg.vocab_size)
                if c >= cfg.min_count]

        self.word2idx = {w: i for i, (w, _) in enumerate(kept)}
        self.idx2word = {i: w for w, i in self.word2idx.items()}
        self.counts   = np.array([c for _, c in kept], dtype=np.float64)
        self.size     = len(kept)

        self._build_neg_table()
        corpus = self._encode_and_subsample(tokens)

        print(f"  Vocabulary : {self.size:,} words")
        print(f"  Corpus     : {len(corpus):,} tokens  "
              f"(after OOV filter + subsampling)")
        return corpus

    def _encode_and_subsample(self, tokens: List[str]) -> np.ndarray:
        """
        Convert tokens to indices, then drop tokens with probability
          P_drop(w) = 1 − √(t / f(w))
        This removes frequent but uninformative words.
        """
        corpus = np.array([self.word2idx[w] for w in tokens
                           if w in self.word2idx], dtype=np.int32)
        f      = self.counts / self.counts.sum()
        p_keep = np.sqrt(self.cfg.subsample_t / (f + 1e-10)).clip(0, 1)
        mask   = np.random.rand(len(corpus)) < p_keep[corpus]
        return corpus[mask]

    def _build_neg_table(self):
        """
        Fill a large integer table so that word i appears in proportion to
          freq(i)^neg_power.  Sampling is then a single random index lookup.
        """
        probs  = self.counts ** self.cfg.neg_power
        probs /= probs.sum()
        table  = np.zeros(self.NEG_TABLE_SIZE, dtype=np.int32)
        k, cumulative = 0, 0.0
        for i, p in enumerate(probs):
            cumulative += p
            while k < self.NEG_TABLE_SIZE * cumulative and k < self.NEG_TABLE_SIZE:
                table[k] = i
                k += 1
        self._neg_table = table

    def sample_negatives(self, n: int) -> np.ndarray:
        idx = np.random.randint(0, self.NEG_TABLE_SIZE, size=n)
        return self._neg_table[idx]

    def __len__(self): return self.size

## Optimiser:  Sparse Adam

Standard Adam updates every row of the embedding matrix on every step, even rows that received a zero gradient. That wastes time and, more subtly, incorrectly inflates the bias-correction factor for rows that haven't actually been seen.

This sparse variant tracks moment estimates only for rows that appear in the current batch.

In [5]:
class SparseAdam:
    """
    Adam optimiser for embedding tables.
    This sparse variant only updates the moment estimates (and parameters)
    for the rows that actually appear in the current batch.
    """

    def __init__(self, shape, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.m    = np.zeros(shape, dtype=np.float64)
        self.v    = np.zeros(shape, dtype=np.float64)
        self.lr, self.beta1, self.beta2, self.eps = lr, beta1, beta2, eps
        self.t    = 0

    def step(self, params: np.ndarray, grad: np.ndarray,
             rows: np.ndarray) -> None:
        """
        Update `params[rows]` in-place using the gradient `grad`
        (one gradient row per entry in `rows`; duplicates are summed first).
        """
        self.t += 1
        bc1 = 1.0 - self.beta1 ** self.t
        bc2 = 1.0 - self.beta2 ** self.t

        # Accumulate gradients for duplicate indices before updating moments
        unique, inv = np.unique(rows, return_inverse=True)
        g_agg = np.zeros((len(unique), grad.shape[1]), dtype=np.float64)
        np.add.at(g_agg, inv, grad)

        self.m[unique] = self.beta1 * self.m[unique] + (1 - self.beta1) * g_agg
        self.v[unique] = self.beta2 * self.v[unique] + (1 - self.beta2) * g_agg**2

        update = (self.lr / bc1) * self.m[unique] / (
            np.sqrt(self.v[unique] / bc2) + self.eps)
        np.add.at(params, unique, -update)

## Skip-Gram with Negative Sampling

Given a centre word `c` and a context word `o`, the SGNS objective maximises `log σ(v_c · u_o)` while minimising `log σ(v_c · u_n)` for each noise word `n`.

In matrix form for a batch of B pairs with K negatives each:

```
loss = mean[ −log σ(s_pos) − Σₖ log σ(−s_neg_k) ]
```

We maintain two separate embedding matrices: `W_in` (centre word lookup) and `W_out` (context/noise word lookup). At inference time we average them.

In [6]:
class SkipGramNS:
    """
    Skip-Gram with Negative Sampling.

    Training pipeline:
      1. Generate all (center, context) pairs with a dynamic window.
      2. For each batch, sample K negative words per pair.
      3. Compute the SGNS objective and its exact analytical gradients.
      4. Update W_in and W_out with sparse Adam.

    After training, call `.embeddings` for L2-normalised vectors.
    """

    def __init__(self, vocab: Vocabulary, cfg: Config):
        np.random.seed(cfg.seed)
        self.vocab, self.cfg = vocab, cfg
        V, D = vocab.size, cfg.embed_dim

        # W_in:  small random values break symmetry so words learn differently
        # W_out: zero initialisation
        self.W_in  = (np.random.rand(V, D).astype(np.float64) - 0.5) / D
        self.W_out = np.zeros((V, D), dtype=np.float64)

        self._opt_in  = SparseAdam((V, D), cfg.lr, cfg.beta1, cfg.beta2, cfg.adam_eps)
        self._opt_out = SparseAdam((V, D), cfg.lr, cfg.beta1, cfg.beta2, cfg.adam_eps)

        self.loss_history: List[float] = []

    # ── pair generation ──────────────────────────────────────────────────────

    def _build_pairs(self, corpus: np.ndarray) -> np.ndarray:
        """
        Yield every (center, context) pair using a dynamic window:
        for each center position t, sample w ~ Uniform[1, window] and
        collect all context positions in [t−w, t+w] \ {t}.

        Dynamic windowing gives closer words a higher effective weight
        (they appear in more windows on average)
        """
        pairs = []
        n = len(corpus)
        for t in range(n):
            w = np.random.randint(1, self.cfg.window + 1)
            lo, hi = max(0, t - w), min(n, t + w + 1)
            for j in range(lo, hi):
                if j != t:
                    pairs.append((corpus[t], corpus[j]))
        return np.array(pairs, dtype=np.int32)

    # ── forward + backward for one batch ────────────────────────────────────

    def _step(self, centers: np.ndarray,
              contexts: np.ndarray) -> float:
        """
        Vectorised SGNS forward + backward for one batch.

        Shapes (B = batch size, K = negatives, D = embed_dim):
          centers, contexts : (B,)
          V_c               : (B, D)   center embeddings
          U_pos             : (B, D)   positive context embeddings
          neg_idx           : (B, K)
          U_neg             : (B, K, D)

        Forward:
          s_pos  = diag(V_c @ U_posᵀ)          (B,)
          s_neg  = einsum('bd,bkd->bk', V_c, U_neg)   (B, K)
          loss   = mean[ −log σ(s_pos) − Σₖ log σ(−s_neg_k) ]

        Backward (negatives of the gradients above, because we minimise):
          e_pos  = σ(s_pos) − 1                 (B,)    ∈ (−1, 0]
          e_neg  = σ(s_neg)                     (B, K)  ∈  (0, 1)

          dW_in[c]   = e_pos·u_o + Σₖ e_neg_k·u_{nk}      accumulated per center
          dW_out[o]  = e_pos·v_c                          accumulated per context
          dW_out[nk] = e_neg_k·v_c                        accumulated per negative
        """
        B, K, D = len(centers), self.cfg.k_negatives, self.cfg.embed_dim

        V_c   = self.W_in [centers]                         # (B, D)
        U_pos = self.W_out[contexts]                        # (B, D)

        neg_idx = self.vocab.sample_negatives(B * K).reshape(B, K)
        U_neg   = self.W_out[neg_idx]                       # (B, K, D)

        # ── forward ──────────────────────────────────────────────────────────
        s_pos = np.einsum("bd,bd->b",   V_c, U_pos)        # (B,)
        s_neg = np.einsum("bd,bkd->bk", V_c, U_neg)        # (B, K)

        sig_pos = _sigmoid(s_pos)                           # (B,)
        sig_neg = _sigmoid(s_neg)                           # (B, K)

        loss = (-np.log(sig_pos + 1e-10)
                - np.sum(np.log(1 - sig_neg + 1e-10), axis=1)).mean()

        # ── backward ─────────────────────────────────────────────────────────
        e_pos = (sig_pos - 1.0)[:, None]                   # (B, 1)
        e_neg = sig_neg                                     # (B, K)

        dV_c    = e_pos * U_pos + np.einsum("bk,bkd->bd", e_neg, U_neg)
        dU_pos  = e_pos * V_c                               # (B, D)
        dU_neg  = e_neg[:, :, None] * V_c[:, None, :]      # (B, K, D)

        # ── parameter updates ─────────────────────────────────────────────────
        self._opt_in .step(self.W_in,  dV_c,                centers)
        self._opt_out.step(self.W_out, dU_pos,               contexts)
        self._opt_out.step(self.W_out, dU_neg.reshape(-1, D),
                           neg_idx.reshape(-1))

        return float(loss)

    # ── training loop ────────────────────────────────────────────────────────

    def fit(self, corpus: np.ndarray) -> "SkipGramNS":
        cfg = self.cfg
        print("\nGenerating training pairs …")
        pairs = self._build_pairs(corpus)
        print(f"  {len(pairs):,} pairs\n")

        for epoch in range(1, cfg.epochs + 1):
            pairs = pairs[np.random.permutation(len(pairs))]   # shuffle
            batches = range(0, len(pairs), cfg.batch_size)
            bar = tqdm(batches, desc=f"Epoch {epoch}/{cfg.epochs}",
                       unit="batch", dynamic_ncols=True)

            epoch_loss, n = 0.0, 0
            for start in bar:
                b = pairs[start : start + cfg.batch_size]
                loss = self._step(b[:, 0], b[:, 1])
                epoch_loss += loss
                n += 1
                if n % 200 == 0:
                    bar.set_postfix(loss=f"{epoch_loss/n:.4f}")

            avg = epoch_loss / n
            self.loss_history.append(avg)
            print(f"      epoch {epoch}  avg loss {avg:.4f}")

        return self

    # ── inference ────────────────────────────────────────────────────────────

    @property
    def embeddings(self) -> np.ndarray:
        """L2-normalised average of W_in and W_out."""
        e = (self.W_in + self.W_out) / 2.0
        return e / (np.linalg.norm(e, axis=1, keepdims=True) + 1e-10)

    def most_similar(self, word: str, topn: int = 10) -> List[Tuple[str, float]]:
        if word not in self.vocab.word2idx:
            return []
        idx  = self.vocab.word2idx[word]
        sims = self.embeddings @ self.embeddings[idx]
        sims[idx] = -1.0
        top  = np.argsort(sims)[::-1][:topn]
        return [(self.vocab.idx2word[i], round(float(sims[i]), 4)) for i in top]

    def analogy(self, a: str, b: str, c: str,
                topn: int = 5) -> List[Tuple[str, float]]:
        """Solve a − b + c = ? (3CosAdd)."""
        for w in (a, b, c):
            if w not in self.vocab.word2idx:
                return []
        emb  = self.embeddings
        q    = emb[self.vocab.word2idx[a]] \
             - emb[self.vocab.word2idx[b]] \
             + emb[self.vocab.word2idx[c]]
        q   /= np.linalg.norm(q) + 1e-10
        sims = emb @ q
        for w in (a, b, c):
            sims[self.vocab.word2idx[w]] = -1.0
        top  = np.argsort(sims)[::-1][:topn]
        return [(self.vocab.idx2word[i], round(float(sims[i]), 4)) for i in top]

    def vector(self, word: str) -> Optional[np.ndarray]:
        if word not in self.vocab.word2idx:
            return None
        return self.embeddings[self.vocab.word2idx[word]]

    # ── persistence ──────────────────────────────────────────────────────────

    def save_checkpoint(self, path: str = "model.npz") -> None:
        """
        Save a full training checkpoint so the model can be resumed or reused
        without retraining.

        Usage:
          model.save_checkpoint("model.npz")
          model = SkipGramNS.load_checkpoint("model.npz")
        """
        # Serialise word2idx as two parallel arrays: words and indices
        words   = np.array(list(self.vocab.word2idx.keys()),   dtype=object)
        indices = np.array(list(self.vocab.word2idx.values()), dtype=np.int32)

        np.savez(
            path,
            # ── embeddings ────────────────────────────────────────────────
            W_in  = self.W_in,
            W_out = self.W_out,
            # ── Adam state (opt_in) ───────────────────────────────────────
            opt_in_m  = self._opt_in.m,
            opt_in_v  = self._opt_in.v,
            opt_in_t  = np.array(self._opt_in.t),
            # ── Adam state (opt_out) ──────────────────────────────────────
            opt_out_m = self._opt_out.m,
            opt_out_v = self._opt_out.v,
            opt_out_t = np.array(self._opt_out.t),
            # ── training history ──────────────────────────────────────────
            loss_history = np.array(self.loss_history),
            # ── vocabulary ────────────────────────────────────────────────
            vocab_words   = words,
            vocab_indices = indices,
            vocab_counts  = self.vocab.counts,
            vocab_neg_table = self.vocab._neg_table,
            # ── config (stored as a flat 1-D float/int array + a key list) ─
            cfg_embed_dim   = np.array(self.cfg.embed_dim),
            cfg_window      = np.array(self.cfg.window),
            cfg_k_negatives = np.array(self.cfg.k_negatives),
            cfg_neg_power   = np.array(self.cfg.neg_power),
            cfg_epochs      = np.array(self.cfg.epochs),
            cfg_batch_size  = np.array(self.cfg.batch_size),
            cfg_lr          = np.array(self.cfg.lr),
            cfg_beta1       = np.array(self.cfg.beta1),
            cfg_beta2       = np.array(self.cfg.beta2),
            cfg_adam_eps    = np.array(self.cfg.adam_eps),
            cfg_vocab_size  = np.array(self.cfg.vocab_size),
            cfg_min_count   = np.array(self.cfg.min_count),
            cfg_subsample_t = np.array(self.cfg.subsample_t),
            cfg_seed        = np.array(self.cfg.seed),
        )
        size_mb = os.path.getsize(path + ".npz" if not path.endswith(".npz") else path) / 1e6
        print(f"Checkpoint saved at {path}  ({size_mb:.1f} MB)")
        print(f"  embeddings : W_in {self.W_in.shape}, W_out {self.W_out.shape}")
        print(f"  epochs done: {len(self.loss_history)}")
        print(f"  resume with: model = SkipGramNS.load_checkpoint('{path}')")

    def save_embeddings_txt(self, path: str = "embeddings.txt") -> None:
        """
        Export the final (averaged, L2-normalised) embedding vectors in the
        standard word2vec text format so they can be loaded by gensim or any
        other tool that speaks this format.

        This is a *read-only* export and does not save optimizer state, so
        training cannot be resumed from it. Use save_checkpoint() for that.

        Usage (gensim):
          from gensim.models import KeyedVectors
          wv = KeyedVectors.load_word2vec_format("embeddings.txt", binary=False)
          wv.most_similar("king")
        """
        V, D = self.vocab.size, self.cfg.embed_dim
        raw  = (self.W_in + self.W_out) / 2.0
        with open(path, "w", encoding="utf-8") as f:
            f.write(f"{V} {D}\n")
            for i in range(V):
                row = " ".join(f"{x:.6f}" for x in raw[i])
                f.write(f"{self.vocab.idx2word[i]} {row}\n")
        print(f"Embeddings exported to {path}  (word2vec text format, gensim-compatible)")

    @classmethod
    def load_checkpoint(cls, path: str) -> "SkipGramNS":
        """
        Reconstruct a SkipGramNS instance from a checkpoint saved by
        save_checkpoint().  The returned model is ready for inference
        immediately, and can also be passed back to .fit() to continue
        training from where it left off.

        Example:
          model = SkipGramNS.load_checkpoint("model.npz")
          model.most_similar("king")           # inference
          model.fit(corpus)                    # or resume training
        """
        data = np.load(path, allow_pickle=True)

        # ── rebuild Config ────────────────────────────────────────────────
        cfg = Config(
            embed_dim   = int(data["cfg_embed_dim"]),
            window      = int(data["cfg_window"]),
            k_negatives = int(data["cfg_k_negatives"]),
            neg_power   = float(data["cfg_neg_power"]),
            epochs      = int(data["cfg_epochs"]),
            batch_size  = int(data["cfg_batch_size"]),
            lr          = float(data["cfg_lr"]),
            beta1       = float(data["cfg_beta1"]),
            beta2       = float(data["cfg_beta2"]),
            adam_eps    = float(data["cfg_adam_eps"]),
            vocab_size  = int(data["cfg_vocab_size"]),
            min_count   = int(data["cfg_min_count"]),
            subsample_t = float(data["cfg_subsample_t"]),
            seed        = int(data["cfg_seed"]),
        )

        # ── rebuild Vocabulary (without re-processing corpus) ─────────────
        vocab = Vocabulary(cfg)
        words_arr   = data["vocab_words"]
        indices_arr = data["vocab_indices"]
        vocab.word2idx  = {str(w): int(i) for w, i in zip(words_arr, indices_arr)}
        vocab.idx2word  = {int(i): str(w) for w, i in zip(words_arr, indices_arr)}
        vocab.counts    = data["vocab_counts"]
        vocab.size      = len(vocab.word2idx)
        vocab._neg_table = data["vocab_neg_table"]

        # ── rebuild model ─────────────────────────────────────────────────
        # Bypass __init__ weight initialisation — we're restoring from disk
        model = cls.__new__(cls)
        model.vocab = vocab
        model.cfg   = cfg
        model.W_in  = data["W_in"]
        model.W_out = data["W_out"]
        model.loss_history = data["loss_history"].tolist()

        # ── restore Adam optimiser state ──────────────────────────────────
        V, D = vocab.size, cfg.embed_dim
        model._opt_in  = SparseAdam((V, D), cfg.lr, cfg.beta1, cfg.beta2, cfg.adam_eps)
        model._opt_out = SparseAdam((V, D), cfg.lr, cfg.beta1, cfg.beta2, cfg.adam_eps)

        model._opt_in.m  = data["opt_in_m"]
        model._opt_in.v  = data["opt_in_v"]
        model._opt_in.t  = int(data["opt_in_t"])
        model._opt_out.m = data["opt_out_m"]
        model._opt_out.v = data["opt_out_v"]
        model._opt_out.t = int(data["opt_out_t"])

        epochs_done = len(model.loss_history)
        print(f"Checkpoint loaded ← {path}")
        print(f"  vocab: {vocab.size:,} words  |  embed_dim: {cfg.embed_dim}")
        print(f"  epochs completed: {epochs_done}  |  "
              f"last loss: {model.loss_history[-1]:.4f}" if epochs_done else "  (no training yet)")
        return model


<>:35: SyntaxWarning: invalid escape sequence '\ '
<>:35: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_682/424014467.py:35: SyntaxWarning: invalid escape sequence '\ '
  collect all context positions in [t−w, t+w] \ {t}.


In [7]:
def _sigmoid(x: np.ndarray) -> np.ndarray:
    """Numerically stable sigmoid taht avoids overflow for large |x|."""
    return np.where(x >= 0,
                    1.0 / (1.0 + np.exp(-x)),
                    np.exp(x) / (1.0 + np.exp(x)))

## Gradient check

Before training anything, we verify the backward pass is correct by comparing the analytical gradients against central finite differences:

```
∂L/∂xᵢ ≈ [L(x + εeᵢ) − L(x − εeᵢ)] / 2ε
```


In [8]:
def gradient_check(model: SkipGramNS,
                   center: int = 7,
                   context: int = 42,
                   eps: float = 1e-5) -> None:
    """
    Confirm the analytical gradient matches a finite-difference estimate.

    Central difference: df/dx ≈ [f(x+ε) − f(x−ε)] / (2ε)
    """
    K   = model.cfg.k_negatives
    neg = model.vocab.sample_negatives(K)

    def loss_at(v_c):
        u_o   = model.W_out[context]
        U_neg = model.W_out[neg]
        s_pos = float(u_o @ v_c)
        s_neg = U_neg @ v_c
        return (-np.log(_sigmoid(s_pos)  + 1e-10)
                - np.sum(np.log(1 - _sigmoid(s_neg) + 1e-10)))

    # Analytical
    v_c   = model.W_in[center].copy()
    u_o   = model.W_out[context].copy()
    U_neg = model.W_out[neg].copy()
    e_pos = _sigmoid(float(u_o @ v_c)) - 1.0
    e_neg = _sigmoid(U_neg @ v_c)
    d_analytical = e_pos * u_o + e_neg @ U_neg      # (D,)

    # Numerical (central differences)
    D = len(v_c)
    d_numerical = np.zeros(D)
    for i in range(D):
        vp, vm = v_c.copy(), v_c.copy()
        vp[i] += eps; vm[i] -= eps
        d_numerical[i] = (loss_at(vp) - loss_at(vm)) / (2 * eps)

    rel_err = (np.linalg.norm(d_analytical - d_numerical) /
               (np.linalg.norm(d_analytical) +
                np.linalg.norm(d_numerical) + 1e-10))
    status  = "PASS" if rel_err < 1e-4 else "FAIL"

    print(f"  Gradient check {status}")
    print(f"    relative error : {rel_err:.2e}  (threshold 1e-4)")
    print(f"    max |Δ|        : {np.abs(d_analytical - d_numerical).max():.2e}")
    assert rel_err < 1e-4, "Gradient check failed!"



## Evaluation

**WordSim-353** — a dataset of 353 word pairs rated by humans for semantic similarity (e.g. *tiger / cat* → 7.35 / 10). We measure Spearman ρ between the model's cosine similarities and the human scores. A higher ρ means the model's geometry agrees with human intuition.

In [9]:
def evaluate_wordsim(model: "SkipGramNS", path: str) -> float:
    """Spearman ρ between model cosine similarities and human ratings."""
    from scipy.stats import spearmanr
    model_scores, human_scores = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) < 3: continue
            try:
                w1, w2, score = parts[0].lower(), parts[1].lower(), float(parts[2])
            except ValueError:
                continue
            v1, v2 = model.vector(w1), model.vector(w2)
            if v1 is None or v2 is None: continue
            model_scores.append(float(np.dot(v1, v2)))
            human_scores.append(score)
    if len(model_scores) < 2:
        return float("nan")
    rho, _ = spearmanr(model_scores, human_scores)
    return float(rho)


## Ablation Study

Runs before the full training to answer: *which hyperparameters matter and why?*

Each configuration differs from the proposed **baseline** (D=100, window=5, K=5) in exactly
one dimension, trained for **2 epochs on a 500k-token slice** of text8 so results
are fast but still informative.

### Metrics reported
* **final_loss** — average SGNS loss on last epoch; lower = better density model.
* **cos(king, queen), cos(paris, berlin)** — cosine similarity between two semantically related words for embedding quality.
* **nn_king (top-3)** — nearest neighbours of "king" -  qualitative sanity check.


In [ ]:
def ablation(tokens: List[str], slice_size: int = 500_000) -> List[dict]:
    sweep = [
        # (label,  embed_dim, window, k_negatives)
        ("D=50  (fewer dims)",      50,  5, 5),
        ("D=100 ← baseline",       100,  5, 5),
        ("D=200 (more dims)",       200,  5, 5),
        ("window=2 (narrow)",       100,  2, 5),
        ("window=10 (wide)",        100, 10, 5),
        ("K=1 (few negatives)",     100,  5, 1),
        ("K=15 (many negatives)",   100,  5, 15),
    ]

    print(f"  Corpus slice : {slice_size:,} tokens")
    print(f"  Epochs       : 2 ")
    print(f"  Vocab cap    : 15,000 ")
    print()

    rows = []
    for label, dim, win, k in sweep:
        print(f"   {label}")

        cfg  = Config(embed_dim=dim, window=win, k_negatives=k,
                      epochs=2, vocab_size=15_000, seed=42)
        voc  = Vocabulary(cfg)
        corp = voc.build(tokens)[:slice_size]
        mdl  = SkipGramNS(voc, cfg).fit(corp)

        final_loss = mdl.loss_history[-1]

        # --- semantic coherence probes ---
        def cos(w1, w2):
            v1, v2 = mdl.vector(w1), mdl.vector(w2)
            if v1 is None or v2 is None: return float("nan")
            return round(float(np.dot(v1, v2)), 3)

        kq   = cos("king",  "queen")
        pb   = cos("paris", "berlin")
        nn_k = [w for w, _ in mdl.most_similar("king", topn=3)]

        row = {
            "config":            label,
            "D / win / K":      f"{dim} / {win} / {k}",
            "final_loss":        round(final_loss, 4),
            "cos(king,queen)":   kq,
            "cos(paris,berlin)": pb,
            "nn_king(top3)":    nn_k,
        }
        rows.append(row)

        print(f"    final_loss={final_loss:.4f}  "
              f"cos(king,queen)={kq}  "
              f"cos(paris,berlin)={pb}  "
              f"nn_king={nn_k}")
        print()

    # ── summary table ────────────────────────────────────────────────────
    print("-" * 88)
    hdr = f"{'Config':<26} {'D/win/K':>10} {'loss':>8} {'cos(k,q)':>10} {'cos(p,b)':>10}"
    print(hdr)
    print("-" * 88)
    for r in rows:
        print(f"{r['config']:<26} {r['D / win / K']:>10} "
              f"{r['final_loss']:>8.4f} {r['cos(king,queen)']:>10} "
              f"{r['cos(paris,berlin)']:>10}")
    print("-" * 88)
    return rows

## Visualisation

In [10]:
def plot_loss(history: List[float]):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(range(1, len(history)+1), history, marker="o", lw=2, color="#4C72B0")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Avg loss")
    ax.set_title("SGNS training loss"); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig("loss.png", dpi=150); plt.show()


def plot_tsne(model: SkipGramNS, n: int = 300):
    import matplotlib.pyplot as plt
    from sklearn.manifold import TSNE

    words = [model.vocab.idx2word[i] for i in range(n)]
    vecs  = model.embeddings[:n]
    proj  = TSNE(n_components=2, perplexity=30,
                 random_state=42).fit_transform(vecs)

    fig, ax = plt.subplots(figsize=(14, 10))
    ax.scatter(proj[:, 0], proj[:, 1], s=6, alpha=0.5, color="#4C72B0")
    for i, w in enumerate(words):
        ax.annotate(w, (proj[i, 0], proj[i, 1]), fontsize=6.5, alpha=0.75)
    ax.set_title("Word embeddings — TSNE (top 300 words)")
    ax.axis("off"); plt.tight_layout()
    plt.savefig("tsne.png", dpi=150); plt.show()




In [11]:
SLICE = None

cfg = Config(
    embed_dim   = 100,
    window      = 5,
    k_negatives = 5,
    vocab_size  = 30_000,
    min_count   = 5,
    subsample_t = 1e-5,
    lr          = 0.001,
    epochs      = 4,
    batch_size  = 512,
)

In [12]:
tokens = fetch_text8()
print(f"Raw corpus: {len(tokens):,} tokens\n")

voc    = Vocabulary(cfg)
corpus = voc.build(tokens)[:SLICE]

Raw corpus: 17,005,207 tokens

  Vocabulary : 30,000 words
  Corpus     : 4,230,372 tokens  (after OOV filter + subsampling)


In [13]:
print("\n── Gradient check (before training) ──")
_check_model = SkipGramNS(voc, cfg)
gradient_check(_check_model)


── Gradient check (before training) ──
  Gradient check PASS
    relative error : 0.00e+00  (threshold 1e-4)
    max |Δ|        : 0.00e+00


## Ablation


In [ ]:
print("\n── Ablation study (500k slice, 2 epochs each) ──\n")
ablation_results = ablation(tokens, slice_size=500_000)



── Ablation study (500k slice, 2 epochs each) ──

  Corpus slice : 500,000 tokens
  Epochs       : 2 
  Vocab cap    : 15,000 

   D=50  (fewer dims)
  Vocabulary : 15,000 words
  Corpus     : 3,598,379 tokens  (after OOV filter + subsampling)

Generating training pairs …
  2,999,176 pairs



Epoch 1/2:   0%|                                                                            | 0/5858 [00:00<?,…

      epoch 1  avg loss 2.6947


Epoch 2/2:   0%|                                                                            | 0/5858 [00:00<?,…

      epoch 2  avg loss 2.5181
    final_loss=2.5181  cos(king,queen)=0.913  cos(paris,berlin)=-0.234  nn_king=['married', 'consort', 'viii']

   D=100 ← baseline
  Vocabulary : 15,000 words
  Corpus     : 3,597,588 tokens  (after OOV filter + subsampling)

Generating training pairs …
  2,997,640 pairs



...

      epoch 2  avg loss 1.2350
    final_loss=1.2350  cos(king,queen)=0.769  cos(paris,berlin)=0.54  nn_king=['emperor', 'prince', 'roman']

   K=15 (many negatives)
  Vocabulary : 15,000 words
  Corpus     : 3,594,305 tokens  (after OOV filter + subsampling)

Generating training pairs …
  2,997,640 pairs



Epoch 1/2:   0%|                                                                            | 0/5855 [00:00<?,…

      epoch 1  avg loss 4.0347


Epoch 2/2:   0%|                                                                            | 0/5855 [00:00<?,…

      epoch 2  avg loss 3.5265
    final_loss=3.5265  cos(king,queen)=0.942  cos(paris,berlin)=0.857  nn_king=['vii', 'emperor', 'pope']

----------------------------------------------------------------------------------------
Config                        D/win/K     loss   cos(k,q)   cos(p,b)
----------------------------------------------------------------------------------------
D=50  (fewer dims)         50 / 5 / 5   2.5181      0.913     -0.234
D=100 ← baseline           100 / 5 / 5   2.4966      0.889      0.572
D=200 (more dims)          200 / 5 / 5   2.4496      0.796      0.313
window=2 (narrow)          100 / 2 / 5   2.5234      0.913     -0.243
window=10 (wide)           100 / 10 / 5   2.4293      0.845      0.175
K=1 (few negatives)        100 / 5 / 1   1.2350      0.769       0.54
K=15 (many negatives)      100 / 5 / 15   3.5265      0.942      0.857
----------------------------------------------------------------------------------------


## Main
Changing K to 15 as K=15 wins on both semantic probes by a meaningful margin. Higher loss is expected mathematically.

In [14]:
cfg.k_negatives = 15

In [ ]:
print("\n── Training ──")
model = SkipGramNS(voc, cfg)
model.fit(corpus)
plot_loss(model.loss_history)


── Training ──

Generating training pairs …
  25,392,535 pairs



Epoch 1/4:   0%|          | 0/49595 [00:00<?, ?batch/s]

In [ ]:
print("\n── Nearest neighbours ──")
for word in ["king", "france", "computer", "music", "war", "science"]:
    print(f"  {word:12s} → {model.most_similar(word, topn=5)}")

print("\n── Analogies (a − b + c = ?) ──")
tests = [
    ("king",   "man",    "woman"),
    ("paris",  "france", "germany"),
    ("walked", "walk",   "run"),
    ("big",    "bigger", "small"),
    ("google", "microsoft", "apple"),
]
for a, b, c in tests:
    res = model.analogy(a, b, c, topn=3)
    print(f"  {a} − {b} + {c} = {res}")

plot_tsne(model, n=300)


model.save_checkpoint("model.npz")
model.save_embeddings_txt("embeddings.txt")